In [9]:
import torch
import torch.nn.functional as F

from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [10]:
classifier = pipeline("sentiment-analysis")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [11]:
text = "I love machine learning and artificial intelligence."

result = classifier(text)

result

[{'label': 'POSITIVE', 'score': 0.9995008707046509}]

In [12]:
texts = [
    "I love this movie. It is amazing!",
    "This course is too difficult and boring.",
    "Machine learning is challenging, but very interesting.",
    "I hate waiting for slow software."
]

results = classifier(texts)

for text, result in zip(texts, results):
    print("文本：", text)
    print("预测结果：", result)
    print("-" * 50)

文本： I love this movie. It is amazing!
预测结果： {'label': 'POSITIVE', 'score': 0.9998822212219238}
--------------------------------------------------
文本： This course is too difficult and boring.
预测结果： {'label': 'NEGATIVE', 'score': 0.9997631907463074}
--------------------------------------------------
文本： Machine learning is challenging, but very interesting.
预测结果： {'label': 'POSITIVE', 'score': 0.9988735318183899}
--------------------------------------------------
文本： I hate waiting for slow software.
预测结果： {'label': 'NEGATIVE', 'score': 0.9863698482513428}
--------------------------------------------------


In [13]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [14]:
text = "I really enjoy learning NLP."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

inputs

{'input_ids': tensor([[  101,  1045,  2428,  5959,  4083, 17953,  2361,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [15]:
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("原始文本：", text)
print("tokens：", tokens)
print("token ids：", token_ids)

原始文本： I really enjoy learning NLP.
tokens： ['i', 'really', 'enjoy', 'learning', 'nl', '##p', '.']
token ids： [1045, 2428, 5959, 4083, 17953, 2361, 1012]


In [16]:
with torch.no_grad():
    outputs = model(**inputs)

outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-3.6885,  3.8900]]), hidden_states=None, attentions=None)

In [17]:
logits = outputs.logits

logits

tensor([[-3.6885,  3.8900]])

In [18]:
probs = F.softmax(logits, dim=-1)

probs

tensor([[5.1106e-04, 9.9949e-01]])

In [19]:
predicted_class_id = torch.argmax(probs, dim=-1).item()

predicted_class_id

1

In [20]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

In [21]:
label = model.config.id2label[predicted_class_id]
score = probs[0][predicted_class_id].item()

print("文本：", text)
print("预测类别：", label)
print("置信度：", score)

文本： I really enjoy learning NLP.
预测类别： POSITIVE
置信度： 0.9994889497756958


In [22]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)
    
    predicted_class_id = torch.argmax(probs, dim=-1).item()
    label = model.config.id2label[predicted_class_id]
    score = probs[0][predicted_class_id].item()
    
    return {
        "text": text,
        "label": label,
        "score": score
    }

In [23]:
test_texts = [
    "This book is very useful.",
    "I do not like this product.",
    "The model is hard to understand, but it is powerful.",
    "This is the worst experience I have ever had."
]

for t in test_texts:
    print(predict_sentiment(t))

{'text': 'This book is very useful.', 'label': 'POSITIVE', 'score': 0.9997184872627258}
{'text': 'I do not like this product.', 'label': 'NEGATIVE', 'score': 0.9982056617736816}
{'text': 'The model is hard to understand, but it is powerful.', 'label': 'POSITIVE', 'score': 0.9998395442962646}
{'text': 'This is the worst experience I have ever had.', 'label': 'NEGATIVE', 'score': 0.9997627139091492}


In [24]:
predict_sentiment("我觉得机器学习很有意思")

{'text': '我觉得机器学习很有意思', 'label': 'NEGATIVE', 'score': 0.7986093759536743}

In [25]:
chinese_model_name = "bert-base-chinese"

chinese_tokenizer = AutoTokenizer.from_pretrained(chinese_model_name)

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

In [26]:
chinese_text = "我正在学习自然语言处理和大模型。"

chinese_tokens = chinese_tokenizer.tokenize(chinese_text)
chinese_ids = chinese_tokenizer.convert_tokens_to_ids(chinese_tokens)

print("原始文本：", chinese_text)
print("tokens：", chinese_tokens)
print("token ids：", chinese_ids)

原始文本： 我正在学习自然语言处理和大模型。
tokens： ['我', '正', '在', '学', '习', '自', '然', '语', '言', '处', '理', '和', '大', '模', '型', '。']
token ids： [2769, 3633, 1762, 2110, 739, 5632, 4197, 6427, 6241, 1905, 4415, 1469, 1920, 3563, 1798, 511]


In [27]:
save_dir = "./saved_sentiment_model"

tokenizer.save_pretrained(save_dir)
model.save_pretrained(save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [28]:
local_tokenizer = AutoTokenizer.from_pretrained(save_dir)
local_model = AutoModelForSequenceClassification.from_pretrained(save_dir)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [29]:
text = "This course is very helpful."

inputs = local_tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = local_model(**inputs)

probs = F.softmax(outputs.logits, dim=-1)
predicted_class_id = torch.argmax(probs, dim=-1).item()

print(local_model.config.id2label[predicted_class_id])
print(probs[0][predicted_class_id].item())

POSITIVE
0.9997082352638245


In [30]:
my_texts = [
    "I think AI agents will change software development.",
    "This model is not good enough.",
    "Learning machine learning step by step is useful.",
    "I am confused about pretrained models."
]

for text in my_texts:
    result = predict_sentiment(text)
    print(result)

{'text': 'I think AI agents will change software development.', 'label': 'NEGATIVE', 'score': 0.9986386895179749}
{'text': 'This model is not good enough.', 'label': 'NEGATIVE', 'score': 0.9998040795326233}
{'text': 'Learning machine learning step by step is useful.', 'label': 'POSITIVE', 'score': 0.9992092251777649}
{'text': 'I am confused about pretrained models.', 'label': 'NEGATIVE', 'score': 0.9992189407348633}
